In [8]:
import pandas as pd
from pathlib import Path
import re
from concurrent.futures import ThreadPoolExecutor


In [9]:
import os

print("Number of logical CPUs:", os.cpu_count())


Number of logical CPUs: 12


In [10]:
file_path = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\LinkedIn\dataset_linkedin-post-search-scraper_2025-05-28_19-02-48-745.csv"

# file_path_ex = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\LinkedIn\hiring\CSV\dataset_linkedin-post-search-scraper_2025-05-27_17-29-45-846.csv"
# df_ex = pd.read_csv(file_path_ex)
# print(df_ex.columns.tolist())
# posibole columns:
# 1. canPostComments - מציין אם ניתן להגיב על הפוסט
# 2. canReact - מציין אם ניתן להגיב לפוסט (לייק, אהבה וכו')
# 3. canShare - מציין אם ניתן לשתף את הפוסט
# 4. commentingDisabled - מציין אם ניתן להגיב על הפוסט
# 5. allowedCommentersScope - מציג מי מורשה להגיב על הפוסט
# 6. comments/0/text - תוכן הטקסט של התגובה
# 7. isRepost - מציין אם הפוסט הוא ריסט של פוסט אחר
# 8. numLikes - מספר הלייקים של הפוסט
# 9. numShares - מספר השיתופים של הפוסט
# 10. reactions/1/type - סוג התגובה (לייק, אהבה וכו')
# 11. resharedPost/rootShare - מציין אם הפוסט הוא ריסט של פוסט אחר
# 12. resharedPost/shareAudience - מציין אם הפוסט מושך תגובות מכל קהל
# 13. text - תוכן הטקסט של הפוסט
# 14. resharedPost/type - סוג הפוסט (פוסט, קשר וכו')
# 15. comments/2/text
# 16. comments/3/text
# 17. numComments
# 18. resharedPost/comments/0/text
# 19. resharedPost/comments/1/text
# 20. resharedPost/isRepost
# 21. resharedPost/numComments
# 22. resharedPost/numLikes
# 23. resharedPost/numShares
# 24. resharedPost/text
# 25. rootShare



columns_names_df = pd.read_csv(file_path)

selected_columns = ["canPostComments", "canReact", "canShare", "commentingDisabled", 
                   "allowedCommentersScope", "comments/0/text", "isRepost", 
                   "numLikes", "numShares", "reactions/1/type",
                   "resharedPost/rootShare", "resharedPost/shareAudience", 
                   "text", "resharedPost/canPostComments", "resharedPost/type",
                   "comments/2/text", "comments/3/text", "numComments",
                   "resharedPost/comments/0/text", "resharedPost/comments/1/text", "resharedPost/isRepost",
                    "resharedPost/numComments", "resharedPost/numLikes", "resharedPost/numShares", "resharedPost/text",
                    "rootShare"]

selected_columns_df = columns_names_df.loc[:, selected_columns]

selected_columns_names = selected_columns_df.columns

selected_columns_names

selected_columns_names = selected_columns_df.columns.tolist()

In [11]:
def find_csv_files(folder_root):
    main_folder_path = Path(folder_root)
    if not main_folder_path.exists() or not main_folder_path.is_dir():
        raise ValueError(f"{main_folder_path} not exists")
    
    print(f"{main_folder_path} patent dir")


    csv_files_per_folder = {}

    for dir in main_folder_path.iterdir():
        if dir.is_dir() and re.match(r"[A-Za-z_-]+", dir.name.strip()):
            print(f"Found directory: {dir.name}")
            csv_folder = dir/ "CSV"
            if csv_folder.is_dir():
                csv_files = list(csv_folder.glob("*.csv"))
                csv_files_per_folder[dir.name] = {
                    "files": csv_files,
                    "count": len(csv_files)
                }
                
    # print("csv files per folder:", csv_files_per_folder)
                
            for i, file in enumerate(csv_files):
                print(f"📁 {dir.name} → 📄 {file.name}")
                if i == len(csv_files) - 1:
                    print(f"📁 {dir.name} – סך הכל {len(csv_files)} קבצים\n")

                      
    return csv_files_per_folder 
csv_files_per_folder = find_csv_files("C:/Users/itama/OneDrive - Ariel University/Documents/Desktop/programing/Research/Data Leakage/Data/LinkedIn")


C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\LinkedIn patent dir
Found directory: charity
📁 charity → 📄 dataset_linkedin-post-search-scraper_2025-05-27_17-32-34-443.csv
📁 charity → 📄 dataset_linkedin-post-search-scraper_2025-05-27_17-35-56-135.csv
📁 charity – סך הכל 2 קבצים

Found directory: hiring
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-27_17-29-45-846.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-27_18-07-56-908.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_18-41-20-287.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_18-52-10-299.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_18-56-35-440.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_19-02-48-745.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_19-03-09-845.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_19-03-41-131.csv
📁 hiring → 📄 dataset_l

In [12]:
def merge_csv_files(csv_files_per_folder):

    all_data = []

    for folder, data in csv_files_per_folder.items():
        for file in data["files"]:
            df = pd.read_csv(file)
            existing_columns = [col for col in selected_columns_names if col in df.columns]
            df = df[existing_columns]  # שומר רק את העמודות שקיימות
            all_data.append(df)


    for file in data["files"]:
        file_columns = pd.read_csv(file, nrows=0).columns.tolist()

    # מציאת עמודות חסרות
    missing_columns = [col for col in selected_columns_names if col not in file_columns]

    if missing_columns:
        print(f"⚠️ Warning: Missing columns in {file.name}: {missing_columns}")

    
    merged_df = pd.concat(all_data, ignore_index=True)
    merged_df.to_csv(r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\LinkedIn\linkedin dataset.csv", index=False)
    print("All CSV files have been merged into 'linledin dataset.csv'")
    print("✅ All CSV files have been merged into 'linkedin dataset.csv'")

In [13]:

def create_doc_row(file_path, subject):
    df = pd.read_csv(file_path)
    num_post = len(df)

    if "inputUrl" in df.columns and not df["inputUrl"].isna().all():
        linkedin_url = df["inputUrl"].dropna().iloc[0]
    else:
        linkedin_url = ""

    parameters = ",".join(df.columns)

    return {
        "URL": linkedin_url,
        "Number of posts": num_post,
        "subject": subject,
        "parameters": parameters
    }


def generate_documentation(csv_files_per_folder, output_path="linkedin Scraping documentation.xlsx"):
    task_list = [
        (file, folder)
        for folder, data in csv_files_per_folder.items()
        for file in data["files"]
    ]

    def row_creator(task):
        file_path, subject = task
        return create_doc_row(file_path, subject)

    with ThreadPoolExecutor(max_workers=8) as executor:
        documentation_rows = list(executor.map(row_creator, task_list))

    # output_path = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\Facebook"
    documentation_df = pd.DataFrame(documentation_rows)
    documentation_df.to_excel(r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\LinkedIn\linkedin Scraping documentation.xlsx", index=False)
    print(f"📄 Documentation saved to: {output_path}")


In [14]:
# מימוש בפונקציה

folder_root = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\LinkedIn"

csv_files_per_folder = find_csv_files(folder_root)


merge_csv_files(csv_files_per_folder)

generate_documentation(csv_files_per_folder)


C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\LinkedIn patent dir
Found directory: charity
📁 charity → 📄 dataset_linkedin-post-search-scraper_2025-05-27_17-32-34-443.csv
📁 charity → 📄 dataset_linkedin-post-search-scraper_2025-05-27_17-35-56-135.csv
📁 charity – סך הכל 2 קבצים

Found directory: hiring
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-27_17-29-45-846.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-27_18-07-56-908.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_18-41-20-287.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_18-52-10-299.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_18-56-35-440.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_19-02-48-745.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_19-03-09-845.csv
📁 hiring → 📄 dataset_linkedin-post-search-scraper_2025-05-28_19-03-41-131.csv
📁 hiring → 📄 dataset_l

C:\Users\itama\AppData\Local\Temp\ipykernel_18576\3547982376.py:7: DtypeWarning: Columns (62,63,64,65,66,67,68,89,90,92,93,94,95,96,97,100,101,102,103,104,105,106,126,127,129,130,131,132,133,134,138,143,157,158,160,161,162,163,179,180,182,183,184,185,186,187,191,203,204,206,207,208,209,210,211,220,227,228,230,231,232,233,234,235,237,239,240,242,243,244,245,246,247,249,251,252,254,255,256,257,258,259,261,263,264,266,267,268,269,270,272,274,276,277,278,279,280,281,283,285,286,288,289,290,291,292,293,295,297,298,300,301,302,303,304,305,307,309,310,312,313,314,315,316,317,319,321,322,324,325,326,327,328,329,331,333,334,336,337,338,339,340,341,343,345,346,348,349,350,351,352,353,355,357,358,360,361,362,363,364,365,367,369,370,372,373,374,375,376,377,379,381,383,384,385,386,387,388,390,392,394,395,396,397,398,399,401,403,405,406,407,408,409,410,412,414,416,417,418,419,420,421,423,425,426,428,429,430,431,432,433,435,437,438,440,441,442,443,444,445,447,449,450,452,453,454,455,456,457,459,461,4

⚠️ Warning: Missing columns in dataset_linkedin-post-search-scraper_2025-05-28_19-43-55-765.csv: ['resharedPost/rootShare', 'resharedPost/shareAudience', 'resharedPost/canPostComments', 'resharedPost/type', 'resharedPost/comments/0/text', 'resharedPost/comments/1/text', 'resharedPost/isRepost', 'resharedPost/numComments', 'resharedPost/numLikes', 'resharedPost/numShares', 'resharedPost/text']
All CSV files have been merged into 'linledin dataset.csv'
✅ All CSV files have been merged into 'linkedin dataset.csv'


C:\Users\itama\AppData\Local\Temp\ipykernel_18576\4090421605.py:2: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,13,14,16,17,18,19,20,21,23,24,25,26,27,28,29,30,33,34,35,36,37,38,39,40,43,44,46,50,51,56,58,59,81,82,83,84,85,86,87,89,90,92,93,94,95,96,97,99,100,101,102,103,104,105,106,108,110,111,112,113,114,115,117,119,120,122,123,124,125,126,127,129,146,147,149,150,151,152,153,154,156,162,173,174,176,177,178,179,180,181,183,189,200,202,203,204,205,206,207,209,215,221,226,227,229,230,231,232,233,234,236,242,253,254,256,257,258,259,260,261,263,269,275,280,281,283,284,285,286,287,288,290,296,297,298,300,301,302,303,304,305,306,307,308,309,311,312,313,315,316,317,318,319,320,322,323,325,326,327,328,329,330,332,333,334,335,336,338,339,340,342,343,344,345,346,347,349,351,352,353,354,355,356,358,359,360,361,362,365,366,367,368,373,374,375,378,379,380,383,384,387,390,394,397,401,404,406,407,409,412,414,417,418,419,420,421,422,423,424,425,426,427,428,434,435,436,437,438,439,440,451,452,453,45

📄 Documentation saved to: linkedin Scraping documentation.xlsx
